In [ ]:
import pandas as pd
import os

DATA_PATH = "../data/raw"

files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

print("Number of CSV files:", len(files))
print("\nFiles:")
for f in files:
    print("-", f)

In [ ]:
for file in files:
    path = os.path.join(DATA_PATH, file)
    
    df = pd.read_csv(path, low_memory=False)
    
    print("\n" + "=" * 80)
    print("FILE:", file)
    print("ROWS:", df.shape[0])
    print("COLUMNS:", df.shape[1])
    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())

In [ ]:
for file in files:
    path = os.path.join(DATA_PATH, file)
    df = pd.read_csv(path, low_memory=False)
    
    print("\n" + "=" * 80)
    print("FILE:", file)
    print(df.isnull().sum())

In [ ]:
for file in files:
    path = os.path.join(DATA_PATH, file)
    df = pd.read_csv(path, low_memory=False)
    
    print("\n" + "=" * 80)
    print("FILE:", file)
    display(df.head(3))

In [ ]:
import pandas as pd

objects = pd.read_csv(
    "../data/raw/objects.csv",
    low_memory=False
)

print("Shape:", objects.shape)

print("\nEntity types:")
print(objects["entity_type"].value_counts(dropna=False))

print("\nTop categories:")
print(objects["category_code"].value_counts().head(30))

print("\nStatus:")
print(objects["status"].value_counts(dropna=False))

In [ ]:
print(objects[
    [
        "id",
        "entity_type",
        "name",
        "category_code",
        "status",
        "founded_at",
        "country_code",
        "funding_rounds",
        "funding_total_usd"
    ]
].head(20))

In [ ]:
print("\nMissing values:")
print(
    objects[
        [
            "name",
            "category_code",
            "status",
            "founded_at",
            "funding_rounds",
            "funding_total_usd"
        ]
    ].isnull().sum()
)

In [ ]:
funding = pd.read_csv(
    "../data/raw/funding_rounds.csv",
    low_memory=False
)

print("Shape:", funding.shape)

print("\nFunding round types:")
print(funding["funding_round_type"].value_counts())

print("\nFunding currencies:")
print(funding["raised_currency_code"].value_counts().head(20))

print("\nUSD funding statistics:")
print(funding["raised_amount_usd"].describe())

In [ ]:
print(
    funding[
        [
            "funding_round_id",
            "object_id",
            "funded_at",
            "funding_round_type",
            "funding_round_code",
            "raised_amount_usd",
            "pre_money_valuation_usd",
            "post_money_valuation_usd",
            "is_first_round",
            "is_last_round"
        ]
    ].head(20)
)

In [ ]:
companies = objects[
    (objects["entity_type"] == "Company") &
    (objects["funding_total_usd"] > 0) &
    (objects["funding_rounds"] > 0)
].copy()

print("Eligible companies:", len(companies))

print(companies[
    [
        "id",
        "name",
        "category_code",
        "status",
        "founded_at",
        "funding_rounds",
        "funding_total_usd",
        "first_funding_at",
        "last_funding_at"
    ]
].head(20))

print("\nMissing values:")
print(
    companies[
        [
            "name",
            "category_code",
            "funding_rounds",
            "funding_total_usd",
            "first_funding_at",
            "last_funding_at"
        ]
    ].isnull().sum()
)

In [ ]:
print("USD funding total statistics:")
print(companies["funding_total_usd"].describe())

print("\nFunding rounds statistics:")
print(companies["funding_rounds"].describe())

In [ ]:
print(
    companies["category_code"]
    .value_counts(dropna=False)
    .head(30)
)

In [ ]:
companies["first_funding_at"] = pd.to_datetime(
    companies["first_funding_at"],
    errors="coerce"
)
companies["last_funding_at"] = pd.to_datetime(
    companies["last_funding_at"],
    errors="coerce"
)
companies["funding_year"] = companies["first_funding_at"].dt.year

print(
    companies["funding_year"]
    .value_counts()
    .sort_index()
)

In [ ]:
acquisitions = pd.read_csv(
    "../data/raw/acquisitions.csv",
    low_memory=False
)

ipos = pd.read_csv(
    "../data/raw/ipos.csv",
    low_memory=False
)

print("Acquisitions:", acquisitions.shape)
print("IPOs:", ipos.shape)

print("\nUnique acquired companies:",
      acquisitions["acquired_object_id"].nunique())

print("Unique acquiring companies:",
      acquisitions["acquiring_object_id"].nunique())

print("Unique IPO companies:",
      ipos["object_id"].nunique())

print("\nIPO table head:")
print(
    ipos[
        [
            "object_id",
            "valuation_amount",
            "valuation_currency_code",
            "raised_amount",
            "raised_currency_code",
            "public_at",
            "stock_symbol"
        ]
    ].head(10)
)

print("\nAcquisitions table head:")
print(
    acquisitions[
        [
            "acquiring_object_id",
            "acquired_object_id",
            "price_amount",
            "price_currency_code",
            "acquired_at"
        ]
    ].head(10)
)

In [ ]:
# Cell 15: Identify eligible 2010-2013 companies and check sector distribution
companies["funding_year"] = pd.to_datetime(
    companies["first_funding_at"],
    errors="coerce"
).dt.year

eligible = companies[
    (companies["funding_year"] >= 2010) &
    (companies["funding_year"] <= 2013)
].copy()

eligible["sector"] = eligible["category_code"].fillna("unknown")

print("Eligible 2010-2013 companies:", len(eligible))

print("\nSector distribution:")
print(eligible["sector"].value_counts().head(20))

In [ ]:
# Cell 16: Check funding statistics by sector
selected_sectors = [
    "software",
    "biotech",
    "web",
    "mobile",
    "ecommerce",
    "enterprise",
    "games_video",
    "advertising",
    "hardware",
    "medical"
]

print(
    eligible.groupby("sector")["funding_total_usd"]
    .agg(["count", "min", "median", "mean", "max"])
    .loc[selected_sectors]
)

In [ ]:
# Cell 17: Check overall funding percentiles
print(
    eligible["funding_total_usd"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
# Cell 18: Candidate pool for project selection
selected_sectors = [
    "software",
    "biotech",
    "web",
    "mobile",
    "ecommerce",
    "enterprise",
    "games_video",
    "advertising",
    "hardware",
    "medical"
]

candidate_pool = eligible[
    (eligible["sector"].isin(selected_sectors)) &
    (eligible["funding_total_usd"] <= 100_000_000)
].copy()

print("Candidate pool:", len(candidate_pool))

print("\nSector counts:")
print(candidate_pool["sector"].value_counts())

print("\nFunding statistics:")
print(candidate_pool["funding_total_usd"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90]
))

In [ ]:
# Cell 19: Check the funding bands distribution by sector
candidate_pool["funding_band"] = pd.cut(
    candidate_pool["funding_total_usd"],
    bins=[
        -float("inf"),
        1_500_000,
        6_141_490,
        20_000_000,
        100_000_000
    ],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

print(
    candidate_pool
    .groupby(["sector", "funding_band"], observed=True)
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Cell 20: Create the stratified sample of 100 projects
sample_config = {
    "Low": 2,
    "Medium": 3,
    "High": 3,
    "Very High": 2
}

sampled_projects = []

for sector in selected_sectors:
    sector_data = candidate_pool[
        candidate_pool["sector"] == sector
    ]

    for band, n in sample_config.items():
        band_data = sector_data[
            sector_data["funding_band"] == band
        ]

        sample = band_data.sample(
            n=n,
            random_state=42
        )

        sampled_projects.append(sample)

startup_projects = pd.concat(
    sampled_projects,
    ignore_index=True
)

print("Total projects:", len(startup_projects))

In [ ]:
# Cell 21: Check sector/band distribution of sample
print(
    startup_projects
    .groupby(["sector", "funding_band"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Cell 22: Assign project IDs
startup_projects = startup_projects.reset_index(drop=True)

startup_projects["project_id"] = [
    f"P{i:03d}"
    for i in range(1, len(startup_projects) + 1)
]

In [ ]:
# Cell 23: Add acquisition and IPO information
acquired_companies = set(
    acquisitions["acquired_object_id"].dropna()
)

ipo_companies = set(
    ipos["object_id"].dropna()
)

startup_projects["acquisition"] = (
    startup_projects["id"]
    .isin(acquired_companies)
    .astype(int)
)

startup_projects["ipo"] = (
    startup_projects["id"]
    .isin(ipo_companies)
    .astype(int)
)

In [ ]:
# Cell 24: Create remaining basic variables
startup_projects["investment_cost"] = (
    startup_projects["funding_total_usd"]
)

startup_projects["first_funding_at"] = pd.to_datetime(
    startup_projects["first_funding_at"],
    errors="coerce"
)

startup_projects["last_funding_at"] = pd.to_datetime(
    startup_projects["last_funding_at"],
    errors="coerce"
)

startup_projects["first_funding_year"] = (
    startup_projects["first_funding_at"].dt.year
)

startup_projects["last_funding_year"] = (
    startup_projects["last_funding_at"].dt.year
)

startup_projects["funding_duration_years"] = (
    startup_projects["last_funding_year"] -
    startup_projects["first_funding_year"]
).fillna(0).clip(lower=0)

In [ ]:
# Cell 25: Filter to final columns and rename
final_columns = [
    "project_id",
    "id",
    "name",
    "sector",
    "country_code",
    "status",
    "founded_at",
    "first_funding_at",
    "last_funding_at",
    "first_funding_year",
    "last_funding_year",
    "funding_duration_years",
    "funding_rounds",
    "funding_total_usd",
    "investment_cost",
    "acquisition",
    "ipo",
    "funding_band"
]

startup_projects = startup_projects[final_columns].copy()

startup_projects.rename(
    columns={
        "id": "company_id",
        "name": "startup_name"
    },
    inplace=True
)

In [ ]:
# Cell 26: Validate the dataset basic info
print("Shape:", startup_projects.shape)

print("\nColumns:")
print(startup_projects.columns.tolist())

print("\nMissing values:")
print(startup_projects.isnull().sum())

print("\nDuplicate project IDs:",
      startup_projects["project_id"].duplicated().sum())

print("\nSector distribution:")
print(startup_projects["sector"].value_counts())

print("\nFunding band distribution:")
print(startup_projects["funding_band"].value_counts())

print("\nAcquisition count:",
      startup_projects["acquisition"].sum())

print("IPO count:",
      startup_projects["ipo"].sum())

In [ ]:
# Cell 27: Validate duplicate company IDs and check sample rows
print(
    "Duplicate companies:",
    startup_projects["company_id"].duplicated().sum()
)

print("\nFirst 20 records:")
print(
    startup_projects[
        [
            "project_id",
            "startup_name",
            "sector",
            "investment_cost",
            "funding_rounds",
            "acquisition",
            "ipo"
        ]
    ].head(20)
)

In [ ]:
# Cell 28: Save the dataset to CSV
startup_projects.to_csv(
    "../data/processed/startup_projects.csv",
    index=False
)

print("Saved successfully.")